# WhisperWatch in Google Colab

WhisperWatch scores English Whisper transcripts so a subtitle editor can decide which clips to review first. A clip is labeled high error when its archived word error rate (WER) exceeds 20%. The score is a ranking signal on a 0–1 scale.

For the dataset walkthrough, use **Runtime → Change runtime type → CPU**, then run cells 1–4. Run cell 5 when the model and evaluation procedure are frozen. Live audio is optional.

## 1. Install the classifier dependencies

These are the versions used for the dataset model. Install faster-whisper later if you want live audio.

In [1]:
%pip install -q scikit-learn==1.8.0 pandas==2.3.3 pyarrow==22.0.0 joblib==1.5.3

## 2. Open the project

Run from an existing checkout if you have one. Before the code is pushed, extract a ZIP of the working tree into the Colab Files panel and set `PROJECT_FOLDER` to that folder. After the branch is published, this cell can clone it if GitHub access is available.

In [2]:
import os
import subprocess
from pathlib import Path

PROJECT_FOLDER = Path("Whisperwatch")
REPO_URL = "https://github.com/idotpixel/Whisperwatch.git"
BRANCH = "feature/model-asr"

project = Path.cwd() if Path("src/train.py").is_file() else PROJECT_FOLDER.resolve()
if not project.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(project)], check=True)
if not (project / "src" / "train.py").is_file():
    raise FileNotFoundError("Set PROJECT_FOLDER to the checkout or extracted working tree containing src/train.py.")
os.chdir(project)
print(project)

## 3. Train on Whispered and score development clips

The first run downloads three English train shards and one development shard (about 59 MB total). The cache lives in `data/raw/whispered/`. You can copy the same Parquet files there before running. The code draws 5,000 clips per training shard with seed 42 and uses all eligible English development clips. To reload an existing model, copy the trusted joblib artifact to `models/whisperwatch.joblib` and set `TRAIN_MODEL = False`.

The [pinned Whispered release](https://huggingface.co/datasets/burakaydinofficial/Whispered/tree/25299fd560dde4271f5177118354af5a961b92fd) contains metadata and Common Voice 17 audio pointers. Both sources are CC0-1.0 according to the dataset card. Cite Whispered (burakaydinofficial, 2026) and [Common Voice (Ardila et al., 2020)](https://aclanthology.org/2020.lrec-1.520/).

The official split names are `train`, `dev`, and `test`. The `other` split is excluded because its sentences overlap the evaluation splits. We use the primary archive pipeline (0.2.0), English clips up to 30 seconds, and rows with valid WER. The classifier reads only Whisper segment measurements, Whisper text, and duration. The joblib stores the fitted imputer, scaler, classifier, feature names in order, dataset revision, and review threshold. Development data selects the cutoff that maximizes F1.

In [3]:
from src.inference import load_model
from src.train import train_development

TRAIN_MODEL = True
if TRAIN_MODEL:
    train_development()
artifact = load_model()
print(f"Review threshold = {artifact['review_threshold']:.6f}")

train-00000.parquet: 37,154 of 39,696 rows eligible


train-00013.parquet: 39,936 of 39,936 rows eligible


train-00026.parquet: 40,000 of 40,000 rows eligible


dev-00000.parquet: 16,359 of 16,361 rows eligible


train: 15,000 rows, 1,153 high-error clips
dev: 16,359 rows, 2,110 high-error clips


Dev average precision: 0.5282
Dev F1: 0.5149 at review threshold 0.252107
Saved and reloaded models/whisperwatch.joblib


Saved 16,359 predictions to outputs/dev_predictions.csv
Review threshold = 0.252107


## 4. Score one real archived clip

This uses the same feature extractor and prediction function that the live path calls. A missing measurement is reported as `None` in the result.

In [4]:
import pyarrow.parquet as pq
from src.features import extract_features
from src.inference import predict_from_features
from src.train import DATA_DIR, get_shard

dev_file = get_shard("dev", 0, DATA_DIR)
columns = ["whisper_text", "segments", "duration_s"]
first_batch = next(pq.ParquetFile(dev_file).iter_batches(batch_size=1, columns=columns))
clip = first_batch.to_pylist()[0]
features = extract_features(clip["segments"], clip["whisper_text"], clip["duration_s"])
result = predict_from_features(features)
print(clip["whisper_text"])
print(f"Review score = {result['review_score']:.3f} ({result['review_label']})")

A child on a swing.
Review score = 0.121 (Low predicted risk)


## 5. Generate the official test export

Set `MODEL_FROZEN = True` after development is complete. This step downloads the English test shard (~9 MB) and applies the saved pipeline and threshold.

**Evaluation team.** Read `outputs/dev_predictions.csv` and `outputs/test_predictions.csv`. Join on `clip_id`. Use `true_high_error` as the binary target, `risk_score` for ranking, and lower `avg_logprob` as a native Whisper risk baseline. `wer`, `human_text`, and `whisper_text` support offline error review.

In [5]:
from src.train import predict_test

MODEL_FROZEN = False
if MODEL_FROZEN:
    predict_test()
else:
    print("Held-out scoring is off. Set MODEL_FROZEN = True when ready.")

Held-out scoring is off. Set MODEL_FROZEN = True when ready.


## Optional live audio for the frontend team

In a separate cell, run `%pip install faster-whisper==1.2.1 av==16.0.1`. On Colab, select a GPU and set `os.environ["WHISPERWATCH_ASR_DEVICE"] = "cuda"` before calling ASR to use float16. CPU defaults to float32 and can be slow. Set `WHISPERWATCH_ASR_COMPUTE_TYPE` to `int8` to opt into CPU quantization. The first run downloads the Large V3 weights (about 3 GB).

Pass a local path to `src.asr.analyze_audio` for English clips up to 30 seconds. Its result contains `transcript`, `review_score`, `review_label`, and `details`. The archive and live runner expose the same segment fields. The archive publishes Large V3, faster-whisper/CTranslate2, and float16, with null temperature for the selected primary pipeline. The complete decoding settings are unavailable. Live scores are experimental.

A local CPU smoke test transcribed Whisper’s 11-second JFK test clip with Large V3 in int8 mode. On the Windows test machine, the process needed `CT2_USE_MKL=0` set before importing faster-whisper to avoid an MKL allocation error. For GPU setup, see the [faster-whisper CUDA requirements](https://github.com/SYSTRAN/faster-whisper#gpu).

Once the Gradio module is integrated, run `python app.py` from the project folder or call the team's launch function from this runtime.

In [6]:
AUDIO_PATH = None  # Set to the local path of a short English recording.
if AUDIO_PATH:
    from src.asr import analyze_audio

    print(analyze_audio(AUDIO_PATH))